# Expanded Classification Training (30,000 Records)
Hardware: 2x Tesla T4 GPUs via PyTorch DDP
Effective Batch Size: 16 (2 per-device x 4 accum x 2 GPUs)
Optimizer Steps: 1,875 (1.0 Epoch)
Evaluation: 3-Way Comparative Benchmark on Original & Corrected Benchmarks

In [ ]:
# 1. Environment & Dependencies Setup
!nvidia-smi
!pip install -q bitsandbytes accelerate peft transformers datasets psutil scikit-learn


In [ ]:
# 2. Verify Dataset & Adapter Inputs
import os
from pathlib import Path
print('Listing /kaggle/input contents:')
for p in sorted(Path('/kaggle/input').glob('*')):
    print(' -', p.name)
    if p.is_dir():
        for sub in sorted(list(p.glob('*')))[:10]:
            print('    *', sub.name)

ds_dir = Path('/kaggle/input/offensive-ai-classification-training')
train_file = ds_dir / 'classification_training_30000.jsonl'
val_orig = ds_dir / 'validation_benchmark_original_1000.jsonl'
val_corr = ds_dir / 'validation_benchmark_corrected_1000.jsonl'
boundary_file = ds_dir / 'boundary_contrast_subset_700.jsonl'
init_adapter = ds_dir / 'initial_adapter'

print(f'Train exists:     {train_file.exists()} ({train_file.stat().st_size:,} bytes)' if train_file.exists() else 'Train missing!')
print(f'Val Orig exists:  {val_orig.exists()} ({val_orig.stat().st_size:,} bytes)' if val_orig.exists() else 'Val Orig missing!')
print(f'Val Corr exists:  {val_corr.exists()} ({val_corr.stat().st_size:,} bytes)' if val_corr.exists() else 'Val Corr missing!')
print(f'Boundary exists:  {boundary_file.exists()} ({boundary_file.stat().st_size:,} bytes)' if boundary_file.exists() else 'Boundary missing!')
print(f'Adapter exists:   {init_adapter.exists()}' if init_adapter.exists() else 'Initial Adapter missing!')


In [ ]:
# 3. Write Neutral Expanded Training Script
script_file = Path('/kaggle/working/train_expanded_classification.py')
script_file.write_text("\"\"\"\ntrain_expanded_classification.py - 2x Tesla T4 GPU Training Script (30,000 Expanded Records)\nConfiguration:\n1. Model Initialization: Initial Adapter (saved from initial training) on top of Qwen2.5-1.5B-Instruct\n2. Training Dataset: classification_training_30000.jsonl (30,000 records, 100% unique, zero leak)\n3. Validation Benchmarks:\n   - Original Benchmark (1,000 records, 125/class)\n   - Corrected Benchmark (1,000 records, 52 setup traffic relabeled to Benign)\n   - Boundary Contrast Diagnostic Subset (700 records: 350 XSS / 350 RCE)\n4. Training Scope: Exactly 1 epoch (1,875 optimizer steps) at learning rate 5e-5\n5. Hardware: 2x Tesla T4 GPUs via PyTorch DDP (torchrun --nproc_per_node=2)\n6. Throughput Optimization: Dynamic sequence padding, group_by_length=True, effective batch size 16\n7. Post-Training Evaluation: 3-way comparative evaluation across Baseline, Initial, and Expanded models.\n\"\"\"\n\nimport os\nimport sys\nimport time\nimport json\nimport psutil\nimport inspect\nfrom pathlib import Path\nfrom collections import Counter, defaultdict\nimport torch\nimport torch.distributed as dist\nfrom datasets import Dataset\nfrom peft import PeftModel, LoraConfig, get_peft_model, prepare_model_for_kbit_training\nfrom transformers import (\n    AutoTokenizer,\n    AutoModelForCausalLM,\n    BitsAndBytesConfig,\n    Trainer,\n    TrainingArguments,\n    TrainerCallback\n)\nimport peft.utils.save_and_load as _peft_save_load\n\ndef _ddp_skip_tensor_parallel_sharding(model, state_dict, adapter_name):\n    for module in model.modules():\n        getter = getattr(module, \"get_base_layer\", None)\n        if getter is None:\n            continue\n        base_layer = getter()\n        if (\n            getattr(base_layer, \"_hf_tp_plan\", None) is not None\n            and getattr(base_layer, \"_hf_device_mesh\", None) is not None\n        ):\n            raise RuntimeError(\"DDP workaround is only valid without tensor parallelism.\")\n    return None\n\n_peft_save_load._maybe_shard_state_dict_for_tp = _ddp_skip_tensor_parallel_sharding\n\n# Paths Configuration\nBASE_MODEL = \"Qwen/Qwen2.5-1.5B-Instruct\"\n\nif Path(\"/kaggle/working\").exists():\n    ROOT = Path(\"/kaggle/working\")\n    INPUT_DIR = Path(\"/kaggle/input\")\n    \n    # Baseline Adapter\n    BASELINE_ADAPTER = INPUT_DIR / \"baseline-checkpoints\" / \"checkpoint-200\"\n    if not BASELINE_ADAPTER.exists():\n        BASELINE_ADAPTER = INPUT_DIR / \"rebuiltv4-training-checkpoints\" / \"checkpoint-200\"\n    if not BASELINE_ADAPTER.exists():\n        BASELINE_ADAPTER = INPUT_DIR / \"rebuiltv4-adapters\" / \"checkpoint-200\"\n        \n    # Initial Adapter (input source for Stage 2)\n    ds_dir = INPUT_DIR / \"offensive-ai-classification-training\"\n    INITIAL_ADAPTER = ds_dir / \"initial_adapter\"\n    if not INITIAL_ADAPTER.exists():\n        INITIAL_ADAPTER = INPUT_DIR / \"initial_training_output\" / \"initial_training_checkpoint_final\"\n        \n    TRAIN_PATH = ds_dir / \"classification_training_30000.jsonl\"\n    VAL_ORIG_PATH = ds_dir / \"validation_benchmark_original_1000.jsonl\"\n    VAL_CORR_PATH = ds_dir / \"validation_benchmark_corrected_1000.jsonl\"\n    BOUNDARY_PATH = ds_dir / \"boundary_contrast_subset_700.jsonl\"\n    \n    OUTPUT_DIR = ROOT / \"classification_training_output\"\nelse:\n    ROOT = Path(\".\")\n    BASELINE_ADAPTER = ROOT / \"staging_baseline_checkpoints\" / \"checkpoint-200\"\n    INITIAL_ADAPTER = ROOT / \"evaluation\" / \"stage1_initial_training\" / \"initial_training_output\" / \"initial_training_checkpoint_final\"\n    TRAIN_PATH = ROOT / \"dataset\" / \"splits\" / \"classification_training_30000.jsonl\"\n    VAL_ORIG_PATH = ROOT / \"dataset\" / \"splits\" / \"validation_benchmark_original_1000.jsonl\"\n    VAL_CORR_PATH = ROOT / \"dataset\" / \"splits\" / \"validation_benchmark_corrected_1000.jsonl\"\n    BOUNDARY_PATH = ROOT / \"dataset\" / \"splits\" / \"boundary_contrast_subset_700.jsonl\"\n    OUTPUT_DIR = ROOT / \"classification_training_output\"\n\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\nMAX_LENGTH = 1024\n\n# Distributed Setup\nis_distributed = \"LOCAL_RANK\" in os.environ or \"RANK\" in os.environ\nif is_distributed:\n    local_rank = int(os.environ.get(\"LOCAL_RANK\", 0))\n    global_rank = int(os.environ.get(\"RANK\", 0))\n    world_size = int(os.environ.get(\"WORLD_SIZE\", 1))\n    torch.cuda.set_device(local_rank)\n    if not dist.is_initialized():\n        dist.init_process_group(backend=\"nccl\")\nelse:\n    local_rank = 0\n    global_rank = 0\n    world_size = 1\n\ndef log_rank0(msg):\n    if global_rank == 0:\n        print(f\"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}\", flush=True)\n\nlog_rank0(\"=\" * 80)\nlog_rank0(f\"Starting Expanded Classification Training (30,000 Records)\")\nlog_rank0(f\"Distributed: {is_distributed} | Local Rank: {local_rank} | World Size: {world_size}\")\nlog_rank0(f\"Base Model:        {BASE_MODEL}\")\nlog_rank0(f\"Initial Adapter:   {INITIAL_ADAPTER}\")\nlog_rank0(f\"Training Dataset:  {TRAIN_PATH}\")\nlog_rank0(f\"Val Benchmark (Orig): {VAL_ORIG_PATH}\")\nlog_rank0(f\"Val Benchmark (Corr): {VAL_CORR_PATH}\")\nlog_rank0(f\"Boundary Dataset:     {BOUNDARY_PATH}\")\nlog_rank0(\"=\" * 80)\n\n# Tokenizer\ntokenizer = AutoTokenizer.from_pretrained(\n    str(INITIAL_ADAPTER) if INITIAL_ADAPTER.exists() else BASE_MODEL,\n    trust_remote_code=True\n)\nif tokenizer.pad_token is None:\n    tokenizer.pad_token = tokenizer.eos_token\ntokenizer.padding_side = \"right\"\n\n# 4-bit Quantization Config\nbnb_config = BitsAndBytesConfig(\n    load_in_4bit=True,\n    bnb_4bit_quant_type=\"nf4\",\n    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,\n    bnb_4bit_use_double_quant=True,\n)\n\n# Load Training Data\nlog_rank0(f\"Loading 30,000 classification training samples...\")\nwith open(TRAIN_PATH, \"r\", encoding=\"utf-8\") as f:\n    train_records = [json.loads(line) for line in f if line.strip()]\n\nlog_rank0(f\"Loaded {len(train_records)} training records. Tokenizing...\")\n\ndef format_records(records):\n    formatted = []\n    classes = [\n        \"Benign\", \"Cookie Injection\", \"Directory Traversal\", \"LOG4J\",\n        \"Log Forging\", \"RCE\", \"SQL Injection\", \"XSS\"\n    ]\n    for r in records:\n        msgs = r[\"messages\"]\n        prompt_text = tokenizer.apply_chat_template(msgs[:2], tokenize=False, add_generation_prompt=True)\n        response_text = msgs[2][\"content\"].strip() + tokenizer.eos_token\n        full_text = prompt_text + response_text\n        \n        tokenized_prompt = tokenizer(prompt_text, add_special_tokens=False)[\"input_ids\"]\n        tokenized_full = tokenizer(full_text, add_special_tokens=False)[\"input_ids\"]\n        \n        if len(tokenized_full) > MAX_LENGTH:\n            tokenized_full = tokenized_full[:MAX_LENGTH]\n            \n        labels = list(tokenized_full)\n        prompt_len = min(len(tokenized_prompt), len(labels))\n        for i in range(prompt_len):\n            labels[i] = -100\n            \n        formatted.append({\n            \"input_ids\": tokenized_full,\n            \"labels\": labels,\n            \"length\": len(tokenized_full)\n        })\n    return formatted\n\ntrain_data = format_records(train_records)\ntrain_dataset = Dataset.from_list(train_data)\nlog_rank0(f\"Tokenized {len(train_dataset)} records.\")\n\n# Data Collator for dynamic padding\nclass DataCollatorForDynamicPadding:\n    def __init__(self, pad_token_id, pad_label_id=-100):\n        self.pad_token_id = pad_token_id\n        self.pad_label_id = pad_label_id\n\n    def __call__(self, features):\n        batch_lens = [len(f[\"input_ids\"]) for f in features]\n        max_batch_len = max(batch_lens)\n        \n        batch_input_ids = []\n        batch_labels = []\n        batch_attention_mask = []\n\n        for f in features:\n            ids = f[\"input_ids\"]\n            labs = f[\"labels\"]\n            pad_len = max_batch_len - len(ids)\n            \n            padded_ids = ids + [self.pad_token_id] * pad_len\n            padded_labs = labs + [self.pad_label_id] * pad_len\n            attn_mask = [1] * len(ids) + [0] * pad_len\n\n            batch_input_ids.append(padded_ids)\n            batch_labels.append(padded_labs)\n            batch_attention_mask.append(attn_mask)\n\n        return {\n            \"input_ids\": torch.tensor(batch_input_ids, dtype=torch.long),\n            \"labels\": torch.tensor(batch_labels, dtype=torch.long),\n            \"attention_mask\": torch.tensor(batch_attention_mask, dtype=torch.long),\n        }\n\ncollator = DataCollatorForDynamicPadding(tokenizer.pad_token_id)\n\n# Model Loading & Adapter Initialization\nlog_rank0(f\"Loading Base Model: {BASE_MODEL}...\")\ndevice_map = {\"\": local_rank} if is_distributed else ({\"\": 0} if torch.cuda.is_available() else None)\n\nbase_model = AutoModelForCausalLM.from_pretrained(\n    BASE_MODEL,\n    quantization_config=bnb_config,\n    device_map=device_map,\n    trust_remote_code=True\n)\nbase_model = prepare_model_for_kbit_training(base_model)\n\nlog_rank0(f\"Initializing adapter from Initial Adapter: {INITIAL_ADAPTER}...\")\nmodel = PeftModel.from_pretrained(\n    base_model,\n    str(INITIAL_ADAPTER),\n    is_trainable=True\n)\nmodel.print_trainable_parameters()\n\n# Dynamic Padding Telemetry Callback\nclass DynamicPaddingTelemetry(TrainerCallback):\n    def __init__(self, max_length):\n        self.max_length = max_length\n        self.total_tokens = 0\n        self.total_padded_tokens = 0\n\n    def on_step_end(self, args, state, control, **kwargs):\n        pass\n\n# Training Arguments\ntrain_args_kwargs = {\n    \"output_dir\": str(OUTPUT_DIR),\n    \"learning_rate\": 5e-5,\n    \"lr_scheduler_type\": \"cosine\",\n    \"warmup_ratio\": 0.05,\n    \"num_train_epochs\": 1.0,\n    \"per_device_train_batch_size\": 2,\n    \"gradient_accumulation_steps\": 4,\n    \"bf16\": torch.cuda.is_bf16_supported(),\n    \"fp16\": not torch.cuda.is_bf16_supported() and torch.cuda.is_available(),\n    \"logging_steps\": 25,\n    \"save_strategy\": \"steps\",\n    \"save_steps\": 250,\n    \"save_total_limit\": 3,\n    \"group_by_length\": True,\n    \"length_column_name\": \"length\",\n    \"report_to\": \"none\",\n    \"gradient_checkpointing\": True,\n    \"dataloader_drop_last\": False,\n    \"ddp_find_unused_parameters\": False if is_distributed else None,\n}\n\nvalid_args = inspect.signature(TrainingArguments.__init__).parameters\nfiltered_args = {k: v for k, v in train_args_kwargs.items() if k in valid_args and v is not None}\ntraining_args = TrainingArguments(**filtered_args)\n\nlog_rank0(\"Instantiating Trainer...\")\ntrainer = Trainer(\n    model=model,\n    args=training_args,\n    train_dataset=train_dataset,\n    data_collator=collator,\n)\n\n# Execute Training\nlog_rank0(\"Starting 30,000-sample training run (1.0 epoch, 1,875 optimizer steps)...\")\nt_start = time.time()\ntrain_result = trainer.train()\ntrain_duration = time.time() - t_start\nsamples_per_sec = len(train_dataset) / train_duration if train_duration > 0 else 0.0\n\nlog_rank0(\"=\" * 80)\nlog_rank0(f\"TRAINING COMPLETE in {train_duration:.2f}s ({train_duration/60:.2f} min)\")\nlog_rank0(f\"Throughput: {samples_per_sec:.3f} samples/sec\")\nlog_rank0(f\"Global Steps: {train_result.global_step}\")\nlog_rank0(f\"Final Train Loss: {train_result.training_loss:.4f}\")\nlog_rank0(\"=\" * 80)\n\n# Hardware Telemetry\npeak_vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024) if torch.cuda.is_available() else 0.0\npeak_vram_per_gpu = []\nif torch.cuda.is_available():\n    for d in range(torch.cuda.device_count()):\n        peak_vram_per_gpu.append(round(torch.cuda.max_memory_allocated(d) / (1024 * 1024), 2))\nsys_ram_gb = psutil.Process().memory_info().rss / (1024 * 1024 * 1024)\n\n# Save Final Trained Adapter\nfinal_adapter_dir = OUTPUT_DIR / \"classification_training_checkpoint_final\"\nif global_rank == 0:\n    log_rank0(f\"Saving final trained adapter to {final_adapter_dir}...\")\n    trainer.save_model(str(final_adapter_dir))\n    tokenizer.save_pretrained(str(final_adapter_dir))\n    log_rank0(\"Final adapter saved successfully.\")\n\n# DDP Collective Synchronization\nif is_distributed:\n    dist.barrier()\n    log_rank0(\"Collective DDP synchronization complete.\")\n\n# Comprehensive Evaluation on Rank 0\nif global_rank == 0:\n    log_rank0(\"\\n\" + \"=\" * 80)\n    log_rank0(\"STARTING 3-WAY MULTI-BENCHMARK COMPARATIVE EVALUATION\")\n    log_rank0(\"=\" * 80)\n    \n    del model\n    del trainer\n    del base_model\n    import gc\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n    # Evaluation Helper\n    classes = [\n        \"Benign\", \"Cookie Injection\", \"Directory Traversal\", \"LOG4J\",\n        \"Log Forging\", \"RCE\", \"SQL Injection\", \"XSS\"\n    ]\n    \n    def evaluate_model_on_dataset(eval_model, dataset_path, eval_name):\n        log_rank0(f\"Evaluating {eval_name} on {dataset_path.name}...\")\n        with open(dataset_path, \"r\", encoding=\"utf-8\") as f:\n            raw_recs = [json.loads(line) for line in f if line.strip()]\n            \n        y_true = []\n        y_pred = []\n        t0 = time.time()\n        \n        for idx, rec in enumerate(raw_recs):\n            y_true.append(rec[\"category\"])\n            prompt_msgs = [rec[\"messages\"][0], rec[\"messages\"][1]]\n            prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)\n            inputs = tokenizer(prompt_text, return_tensors=\"pt\").to(eval_model.device)\n            \n            with torch.no_grad():\n                gen_out = eval_model.generate(\n                    **inputs,\n                    max_new_tokens=15,\n                    pad_token_id=tokenizer.pad_token_id,\n                    eos_token_id=tokenizer.eos_token_id,\n                    do_sample=False\n                )\n            pred_text = tokenizer.decode(gen_out[0][inputs[\"input_ids\"].shape[1]:], skip_special_tokens=True).strip()\n            \n            matched = \"Unknown\"\n            for c in classes:\n                if c.lower() in pred_text.lower():\n                    matched = c\n                    break\n            if matched == \"Unknown\":\n                matched = pred_text\n            y_pred.append(matched)\n\n        dur = time.time() - t0\n        correct = sum(1 for yt, yp in zip(y_true, y_pred) if yt == yp)\n        acc = correct / len(y_true) if y_true else 0.0\n\n        per_class = {}\n        f1_list = []\n        weighted_f1_sum = 0\n        for c in classes:\n            tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == c and yp == c)\n            fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt != c and yp == c)\n            fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == c and yp != c)\n            supp = sum(1 for yt in y_true if yt == c)\n\n            prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0\n            rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0\n            f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0\n            if supp > 0:\n                f1_list.append(f1)\n                weighted_f1_sum += (f1 * supp)\n\n            per_class[c] = {\n                \"precision\": round(prec, 4),\n                \"recall\": round(rec, 4),\n                \"f1_score\": round(f1, 4),\n                \"support\": supp\n            }\n\n        macro_f1 = sum(f1_list) / len(f1_list) if f1_list else 0.0\n        weighted_f1 = weighted_f1_sum / len(y_true) if y_true else 0.0\n\n        confusion_mat = {c_true: {c_pred: 0 for c_pred in classes + [\"Unknown\"]} for c_true in classes}\n        for yt, yp in zip(y_true, y_pred):\n            pred_col = yp if yp in classes else \"Unknown\"\n            if yt in confusion_mat:\n                confusion_mat[yt][pred_col] += 1\n\n        return {\n            \"accuracy\": round(acc, 4),\n            \"macro_f1\": round(macro_f1, 4),\n            \"weighted_f1\": round(weighted_f1, 4),\n            \"per_class\": per_class,\n            \"confusion_matrix\": confusion_mat,\n            \"duration_seconds\": round(dur, 2)\n        }\n\n    def load_model_with_adapter(adapter_path):\n        b = AutoModelForCausalLM.from_pretrained(\n            BASE_MODEL,\n            quantization_config=bnb_config,\n            device_map={\"\": 0} if torch.cuda.is_available() else None,\n            trust_remote_code=True\n        )\n        m = PeftModel.from_pretrained(b, str(adapter_path))\n        m.eval()\n        return m\n\n    # 1. Evaluate Newly Trained Expanded Model\n    log_rank0(f\"Loading Newly Trained Model from {final_adapter_dir}...\")\n    trained_model = load_model_with_adapter(final_adapter_dir)\n    trained_orig_val = evaluate_model_on_dataset(trained_model, VAL_ORIG_PATH, \"Expanded_Model_on_Original_Val\")\n    trained_corr_val = evaluate_model_on_dataset(trained_model, VAL_CORR_PATH, \"Expanded_Model_on_Corrected_Val\")\n    trained_boundary = evaluate_model_on_dataset(trained_model, BOUNDARY_PATH, \"Expanded_Model_on_Boundary_Subset\")\n    del trained_model\n    gc.collect()\n    torch.cuda.empty_cache()\n\n    # 2. Evaluate Initial Adapter Model\n    log_rank0(f\"Loading Initial Adapter Model from {INITIAL_ADAPTER}...\")\n    init_model = load_model_with_adapter(INITIAL_ADAPTER)\n    init_orig_val = evaluate_model_on_dataset(init_model, VAL_ORIG_PATH, \"Initial_Model_on_Original_Val\")\n    init_corr_val = evaluate_model_on_dataset(init_model, VAL_CORR_PATH, \"Initial_Model_on_Corrected_Val\")\n    init_boundary = evaluate_model_on_dataset(init_model, BOUNDARY_PATH, \"Initial_Model_on_Boundary_Subset\")\n    del init_model\n    gc.collect()\n    torch.cuda.empty_cache()\n\n    # 3. Evaluate Baseline Adapter Model\n    log_rank0(f\"Loading Baseline Adapter Model from {BASELINE_ADAPTER}...\")\n    base_eval_model = load_model_with_adapter(BASELINE_ADAPTER)\n    base_orig_val = evaluate_model_on_dataset(base_eval_model, VAL_ORIG_PATH, \"Baseline_Model_on_Original_Val\")\n    base_corr_val = evaluate_model_on_dataset(base_eval_model, VAL_CORR_PATH, \"Baseline_Model_on_Corrected_Val\")\n    base_boundary = evaluate_model_on_dataset(base_eval_model, BOUNDARY_PATH, \"Baseline_Model_on_Boundary_Subset\")\n    del base_eval_model\n    gc.collect()\n    torch.cuda.empty_cache()\n\n    # Compile Final Deliverable Report\n    final_report = {\n        \"status\": \"COMPLETED\",\n        \"experiment\": \"EXPANDED_CLASSIFICATION_TRAINING_30000\",\n        \"training_records\": len(train_dataset),\n        \"epochs\": 1.0,\n        \"optimizer_steps\": train_result.global_step,\n        \"train_loss\": round(train_result.training_loss, 4),\n        \"train_duration_seconds\": round(train_duration, 2),\n        \"measured_throughput_samples_per_sec\": round(samples_per_sec, 3),\n        \"peak_vram_mb\": peak_vram_mb,\n        \"peak_vram_per_gpu\": peak_vram_per_gpu,\n        \"system_ram_gb\": round(sys_ram_gb, 2),\n        \"saved_adapter_path\": str(final_adapter_dir),\n        \"comparative_benchmarks\": {\n            \"original_validation_benchmark_1000\": {\n                \"baseline_model\": base_orig_val,\n                \"initial_model\": init_orig_val,\n                \"expanded_model\": trained_orig_val\n            },\n            \"corrected_validation_benchmark_1000\": {\n                \"baseline_model\": base_corr_val,\n                \"initial_model\": init_corr_val,\n                \"expanded_model\": trained_corr_val\n            },\n            \"boundary_contrast_diagnostic_700\": {\n                \"baseline_model\": base_boundary,\n                \"initial_model\": init_boundary,\n                \"expanded_model\": trained_boundary\n            }\n        }\n    }\n\n    report_path = OUTPUT_DIR / \"classification_training_metrics_report.json\"\n    with open(report_path, \"w\", encoding=\"utf-8\") as f:\n        json.dump(final_report, f, indent=2)\n\n    log_rank0(\"\\n\" + \"=\" * 80)\n    log_rank0(\"3-WAY COMPARATIVE BENCHMARK SUMMARY (ACCURACY / MACRO F1):\")\n    log_rank0(\"1. Original Benchmark (1,000 samples, 125/class):\")\n    log_rank0(f\"   Baseline Adapter: {base_orig_val['accuracy']*100:.2f}% | F1: {base_orig_val['macro_f1']:.4f}\")\n    log_rank0(f\"   Initial Adapter:  {init_orig_val['accuracy']*100:.2f}% | F1: {init_orig_val['macro_f1']:.4f}\")\n    log_rank0(f\"   Expanded Adapter: {trained_orig_val['accuracy']*100:.2f}% | F1: {trained_orig_val['macro_f1']:.4f}\")\n    log_rank0(\"2. Corrected Benchmark (1,000 samples, 52 setup traffic relabeled to Benign):\")\n    log_rank0(f\"   Baseline Adapter: {base_corr_val['accuracy']*100:.2f}% | F1: {base_corr_val['macro_f1']:.4f}\")\n    log_rank0(f\"   Initial Adapter:  {init_corr_val['accuracy']*100:.2f}% | F1: {init_corr_val['macro_f1']:.4f}\")\n    log_rank0(f\"   Expanded Adapter: {trained_corr_val['accuracy']*100:.2f}% | F1: {trained_corr_val['macro_f1']:.4f}\")\n    log_rank0(\"=\" * 80)\n    log_rank0(f\"Complete report written to: {report_path}\")\n    log_rank0(\"TRAINING AND 3-WAY MULTI-BENCHMARK EVALUATION FINISHED SUCCESSFULLY.\")\n", encoding='utf-8')
print(f'Wrote {script_file.name}: {script_file.stat().st_size:,} bytes')


In [ ]:
# 4. Execute Expanded Training via PyTorch DDP
import subprocess
import sys
import os
import torch

env = os.environ.copy()
for k in ['RANK', 'LOCAL_RANK', 'WORLD_SIZE', 'MASTER_ADDR', 'MASTER_PORT']:
    env.pop(k, None)
env['CUDA_VISIBLE_DEVICES'] = '0,1'
env['TOKENIZERS_PARALLELISM'] = 'false'
env['OMP_NUM_THREADS'] = '4'

num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f'Detected available GPU count: {num_gpus}')

if num_gpus > 1:
    cmd = [
        'torchrun',
        '--standalone',
        '--nnodes=1',
        f'--nproc_per_node={num_gpus}',
        '/kaggle/working/train_expanded_classification.py'
    ]
else:
    cmd = [sys.executable, '/kaggle/working/train_expanded_classification.py']

print('Executing:', ' '.join(cmd))
res = subprocess.run(cmd, env=env, capture_output=False)
if res.returncode != 0:
    raise RuntimeError(f'Training failed with exit code {res.returncode}')
print('Expanded classification training and evaluation finished successfully!')


In [ ]:
# 5. Display Final Metrics Report & Deliverables
import json
from pathlib import Path

report_path = Path('/kaggle/working/classification_training_output/classification_training_metrics_report.json')
if report_path.exists():
    print('=' * 80)
    print('CLASSIFICATION TRAINING METRICS REPORT:')
    print('=' * 80)
    print(report_path.read_text(encoding='utf-8'))
else:
    print('Warning: Metrics report not found.')

print('\nOutput directory artifacts:')
for p in sorted(Path('/kaggle/working/classification_training_output').glob('**/*')):
    if p.is_file():
        print(f'{p.relative_to("/kaggle/working")}: {p.stat().st_size:,} bytes')
